# Ice Rheology: Deformation Mechanism Maps

**Ge 193 — Glacier Dynamics, Spring 2026**

This notebook generates figures for the *Ice Rheology* lecture, including deformation mechanism maps that show which creep mechanism dominates as a function of stress, temperature, and grain size.

In [ ]:
# ---- Setup: imports and physical constants ----
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

R = 8.314          # J mol^{-1} K^{-1}, gas constant

# ---- Creep parameters from Goldsby & Kohlstedt (2001), Table 5 ----
# Dislocation creep (n=4, r=0)
A0_disl_cold = 4.0e5    # MPa^{-4} s^{-1}, T < 258 K
A0_disl_warm = 6.0e28   # MPa^{-4} s^{-1}, T > 258 K
n_disl = 4.0
Q_disl_cold = 60e3      # J mol^{-1}
Q_disl_warm = 181e3     # J mol^{-1}

# GBS-accommodated basal slip (n=1.8, r=1.4)
A0_gbs_cold = 3.9e-3    # MPa^{-1.8} m^{1.4} s^{-1}, T < 255 K
A0_gbs_warm = 3.0e26    # MPa^{-1.8} m^{1.4} s^{-1}, T > 255 K
n_gbs = 1.8
r_gbs = 1.4
Q_gbs_cold = 49e3       # J mol^{-1}
Q_gbs_warm = 192e3      # J mol^{-1}

# Basal-slip-accommodated GBS (n=2.4, r=0)
A0_basal = 5.5e7        # MPa^{-2.4} s^{-1}
n_basal = 2.4
Q_basal = 60e3          # J mol^{-1}

# Diffusion creep parameters (Eq. 4 of G&K 2001, Table 6)
V_m = 1.97e-5           # m^3 mol^{-1}, molar volume
D_0v = 9.10e-4          # m^2 s^{-1}, volume diffusion preexponential (Ramseier 1967)
Q_v = 59.4e3            # J mol^{-1}, volume diffusion activation energy
delta = 9.04e-10        # m, grain boundary width (= 2b, Frost & Ashby 1982)
D_0b = 8.4e-4           # m^2 s^{-1}, boundary diffusion preexponential (G&K 2001 upper bound, p. 11026)
Q_b = 49e3              # J mol^{-1}, boundary diffusion activation energy

print('Setup complete.')

In [ ]:
# ---- Strain-rate functions for each mechanism ----

def edot_disl(tau, T):
    """Dislocation creep strain rate. tau in MPa, T in K."""
    A0 = np.where(T < 258, A0_disl_cold, A0_disl_warm)
    Q = np.where(T < 258, Q_disl_cold, Q_disl_warm)
    return A0 * tau**n_disl * np.exp(-Q / (R * T))


def edot_gbs(tau, T, d):
    """GBS-accommodated basal slip strain rate. tau in MPa, d in m, T in K."""
    A0 = np.where(T < 255, A0_gbs_cold, A0_gbs_warm)
    Q = np.where(T < 255, Q_gbs_cold, Q_gbs_warm)
    return A0 * tau**n_gbs / d**r_gbs * np.exp(-Q / (R * T))


def edot_basal(tau, T):
    """Basal-slip-accommodated GBS strain rate. tau in MPa, T in K."""
    return A0_basal * tau**n_basal * np.exp(-Q_basal / (R * T))


def edot_diff(tau, T, d):
    """Diffusional flow strain rate (Nabarro-Herring + Coble). tau in MPa, d in m, T in K."""
    tau_Pa = tau * 1e6  # convert to Pa
    D_v = D_0v * np.exp(-Q_v / (R * T))
    D_b = D_0b * np.exp(-Q_b / (R * T))
    return 42 * tau_Pa * V_m / (R * T * d**2) * (D_v + np.pi * delta / d * D_b)


def edot_composite(tau, T, d):
    """Composite flow law (Eq. 3 of G&K 2001). tau in MPa, d in m, T in K."""
    e_diff = edot_diff(tau, T, d)
    e_basal = edot_basal(tau, T)
    e_gbs = edot_gbs(tau, T, d)
    e_disl = edot_disl(tau, T)
    # Basal slip and GBS in series, rest in parallel
    e_serial = 1.0 / (1.0 / e_basal + 1.0 / e_gbs)
    return e_diff + e_serial + e_disl


def dominant_mechanism(tau, T, d):
    """Return index of dominant mechanism: 0=diff, 1=GBS-accomm basal slip,
    2=basal-slip-accomm GBS, 3=dislocation.
    For the serial pair (basal/GBS), the dominant one is whichever has the
    lower strain rate (i.e., the rate-limiting step)."""
    e_diff = edot_diff(tau, T, d)
    e_basal = edot_basal(tau, T)
    e_gbs = edot_gbs(tau, T, d)
    e_disl = edot_disl(tau, T)
    e_serial = 1.0 / (1.0 / e_basal + 1.0 / e_gbs)
    
    # Effective contributions (parallel sum)
    rates = np.array([e_diff, e_serial, e_disl])
    parallel_dom = np.argmax(rates, axis=0)  # 0=diff, 1=serial, 2=disl
    
    # For the serial mechanism, determine which is rate-limiting
    # GBS-limited -> n=1.8 regime; basal-slip-limited -> n=2.4 regime
    gbs_limited = e_gbs <= e_basal  # GBS is slower => GBS-accommodated basal slip
    
    result = np.zeros_like(tau, dtype=int)
    result[parallel_dom == 0] = 0  # diffusion
    result[(parallel_dom == 1) & gbs_limited] = 1   # GBS-accomm basal slip (n=1.8)
    result[(parallel_dom == 1) & ~gbs_limited] = 2  # basal-slip-accomm GBS (n=2.4)
    result[parallel_dom == 2] = 3  # dislocation creep
    return result


print('Functions defined.')

---
## Figure 1: Strain Rate vs. Stress (fixed $T$ and $d$)

Log-log plot of $\dot{\varepsilon}_e$ vs. $\tau_e$ showing each mechanism individually and the composite flow law. This is the most direct way to see the transitions between regimes and where $n \approx 3$ arises.

In [ ]:
# ---- Fig 1: strain rate vs stress at two grain sizes ----
T_fig1 = 263.0   # K (about -10 C)

tau = np.logspace(-4, 1.5, 500)  # MPa

fig, ax = plt.subplots(figsize=(7, 6))

for d_val, ls, lbl_suffix in [(1e-3, '-', '1 mm'), (10e-3, '--', '10 mm')]:
    e_disl = edot_disl(tau, T_fig1)
    e_gbs = edot_gbs(tau, T_fig1, d_val)
    e_diff = edot_diff(tau, T_fig1, d_val)
    e_total = edot_composite(tau, T_fig1, d_val)

    ax.loglog(tau, e_disl, ls, color='#d62728', lw=1.5, alpha=0.7)
    ax.loglog(tau, e_gbs, ls, color='#2ca02c', lw=1.5, alpha=0.7)
    ax.loglog(tau, e_diff, ls, color='#ff7f0e', lw=1.5, alpha=0.7)
    ax.loglog(tau, e_total, ls, color='k', lw=2.5)

# Shade Glen's experimental stress range
ax.axvspan(0.1, 1.0, alpha=0.08, color='gray')
ax.text(0.3, 2e-14, "Glen's\nexperiments", fontsize=9, ha='center',
        color='gray', style='italic')

# Build legend manually: mechanism colors + line style for grain size
import matplotlib.lines as mlines
leg_disl = mlines.Line2D([], [], color='#d62728', lw=1.5, label=f'Dislocation creep ($n = {n_disl:.0f}$)')
leg_gbs = mlines.Line2D([], [], color='#2ca02c', lw=1.5, label=f'GBS-accomm. basal slip ($n = {n_gbs}$)')
leg_diff = mlines.Line2D([], [], color='#ff7f0e', lw=1.5, label='Diffusional flow ($n = 1$)')
leg_comp = mlines.Line2D([], [], color='k', lw=2.5, label='Composite flow law')
leg_d1 = mlines.Line2D([], [], color='gray', ls='-', lw=1.5, label='$d = 1$ mm')
leg_d10 = mlines.Line2D([], [], color='gray', ls='--', lw=1.5, label='$d = 10$ mm')
ax.legend(handles=[leg_disl, leg_gbs, leg_diff, leg_comp, leg_d1, leg_d10],
          loc='upper left', fontsize=9, frameon=False)

ax.set_xlabel('Effective deviatoric stress $\\tau_e$ (MPa)', fontsize=12)
ax.set_ylabel('Effective strain rate $\\dot{\\varepsilon}_e$ (s$^{-1}$)', fontsize=12)
ax.set_xlim(1e-4, 30)
ax.set_ylim(1e-18, 1e-2)
ax.set_title(f'$T = {T_fig1:.0f}$ K', fontsize=12)

fig.tight_layout()
fig.savefig('figures/fig01_strain_rate_vs_stress.pdf', bbox_inches='tight')
fig.savefig('figures/fig01_strain_rate_vs_stress.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

---
## Figure 2: Deformation Mechanism Map — Stress vs. Temperature

For a fixed grain size, color the $\tau_e$--$T$ plane by the dominant creep mechanism. This is the classic deformation mechanism map.

In [ ]:
# ---- Fig 2: deformation mechanism map (stress vs temperature) ----
d_fig2 = 1e-3  # m (1 mm)

T_arr = np.linspace(200, 273, 400)
tau_arr = np.logspace(-4, 1.5, 400)
T_grid, tau_grid = np.meshgrid(T_arr, tau_arr)

dom = dominant_mechanism(tau_grid, T_grid, d_fig2)

# Colors: 0=diff (orange), 1=GBS-accomm (green), 2=basal-slip-accomm (purple), 3=disl (red)
cmap = ListedColormap(['#ff7f0e', '#2ca02c', '#9467bd', '#d62728'])

fig, ax = plt.subplots(figsize=(8, 6))
ax.pcolormesh(T_arr, tau_arr, dom, cmap=cmap, shading='auto', alpha=0.6)
ax.set_yscale('log')

# Legend patches
labels = ['Diffusional flow ($n=1$)',
          'GBS-accomm. basal slip ($n=1.8$)',
          'Basal-slip-accomm. GBS ($n=2.4$)',
          'Dislocation creep ($n=4$)']
colors = ['#ff7f0e', '#2ca02c', '#9467bd', '#d62728']
patches = [mpatches.Patch(facecolor=c, alpha=0.6, edgecolor='k', lw=0.5, label=l)
           for c, l in zip(colors, labels)]
ax.legend(handles=patches, loc='upper left', fontsize=9, frameon=True,
          facecolor='white', edgecolor='gray')

# Mark typical glacier conditions
ax.axhline(0.1, color='k', ls=':', lw=0.8, alpha=0.5)
ax.text(202, 0.12, '$\\tau_e = 0.1$ MPa (typical glacier)', fontsize=8, alpha=0.7)

ax.set_xlabel('Temperature $T$ (K)', fontsize=12)
ax.set_ylabel('Effective deviatoric stress $\\tau_e$ (MPa)', fontsize=12)
ax.set_title(f'Deformation mechanism map ($d = {d_fig2*1e3:.0f}$ mm)', fontsize=12)
ax.set_xlim(200, 273)
ax.set_ylim(1e-4, 30)

fig.tight_layout()
fig.savefig('figures/fig02_mechanism_map_stress_temp.pdf', bbox_inches='tight')
fig.savefig('figures/fig02_mechanism_map_stress_temp.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

---
## Figure 3: Deformation Mechanism Map — Stress vs. Grain Size

At a fixed temperature, show how grain size controls the transition between grain-size-sensitive (GBS) and grain-size-insensitive (dislocation) creep.

In [ ]:
# ---- Fig 3: deformation mechanism map (stress vs grain size) ----
T_fig3 = 263.0  # K

d_arr = np.logspace(-6, -1, 400)    # m (1 um to 100 mm)
tau_arr3 = np.logspace(-4, 1.5, 400)  # MPa
d_grid, tau_grid3 = np.meshgrid(d_arr, tau_arr3)

dom3 = dominant_mechanism(tau_grid3, T_fig3, d_grid)

fig, ax = plt.subplots(figsize=(8, 6))
ax.pcolormesh(d_arr * 1e3, tau_arr3, dom3, cmap=cmap, shading='auto', alpha=0.6)
ax.set_xscale('log')
ax.set_yscale('log')

ax.legend(handles=patches, loc='lower left', fontsize=9, frameon=True,
          facecolor='white', edgecolor='gray')

# Mark glaciologically relevant grain sizes
ax.axvline(1.0, color='k', ls=':', lw=0.8, alpha=0.5)
ax.text(1.1, 5, '$d = 1$ mm', fontsize=8, alpha=0.7, rotation=90, va='top')
ax.axvline(10.0, color='k', ls=':', lw=0.8, alpha=0.5)
ax.text(11, 5, '$d = 10$ mm', fontsize=8, alpha=0.7, rotation=90, va='top')

ax.set_xlabel('Grain size $d$ (mm)', fontsize=12)
ax.set_ylabel('Effective deviatoric stress $\\tau_e$ (MPa)', fontsize=12)
ax.set_title(f'Deformation mechanism map ($T = {T_fig3:.0f}$ K)', fontsize=12)
ax.set_xlim(1e-3, 100)
ax.set_ylim(1e-4, 30)

fig.tight_layout()
fig.savefig('figures/fig03_mechanism_map_stress_grain.pdf', bbox_inches='tight')
fig.savefig('figures/fig03_mechanism_map_stress_grain.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

---
## Figure 4: Effective Stress Exponent $n_{\mathrm{eff}}$

Show how the local slope of the $\log\dot{\varepsilon}_e$--$\log\tau_e$ curve varies with stress, making concrete the idea that $n = 3$ is a transitional value.

In [ ]:
# ---- Fig 4: effective stress exponent vs stress ----
T_fig4 = 263.0
d_fig4 = 1e-3

tau4 = np.logspace(-4, 1.5, 1000)
log_tau = np.log10(tau4)
log_edot = np.log10(edot_composite(tau4, T_fig4, d_fig4))

# Numerical derivative: n_eff = d(log edot) / d(log tau)
n_eff = np.gradient(log_edot, log_tau)

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.semilogx(tau4, n_eff, 'k-', lw=2)

# Reference lines
ax.axhline(4.0, color='#d62728', ls='--', lw=1, alpha=0.7, label='$n = 4$ (dislocation)')
ax.axhline(3.0, color='gray', ls='--', lw=1, alpha=0.7, label='$n = 3$ (Glen)')
ax.axhline(1.8, color='#2ca02c', ls='--', lw=1, alpha=0.7, label='$n = 1.8$ (GBS-accomm.)')

# Shade Glen's experimental stress range
ax.axvspan(0.1, 1.0, alpha=0.08, color='gray')
ax.text(0.3, 1.3, "Glen's\nexperiments", fontsize=9, ha='center',
        color='gray', style='italic')

ax.set_xlabel('Effective deviatoric stress $\\tau_e$ (MPa)', fontsize=12)
ax.set_ylabel('Effective stress exponent $n_{\\mathrm{eff}}$', fontsize=12)
ax.set_xlim(1e-4, 30)
ax.set_ylim(1.0, 4.5)
ax.legend(loc='center left', fontsize=9, frameon=False)
ax.set_title(f'$T = {T_fig4:.0f}$ K, $d = {d_fig4*1e3:.0f}$ mm', fontsize=12)

fig.tight_layout()
fig.savefig('figures/fig04_effective_n_vs_stress.pdf', bbox_inches='tight')
fig.savefig('figures/fig04_effective_n_vs_stress.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

---
## Figure 5: Effect of Grain Size on the Composite Flow Law

Show how grain size shifts the transition between GBS-dominated and dislocation-dominated flow.

In [ ]:
# ---- Fig 5: composite flow law for different grain sizes ----
T_fig5 = 263.0
tau5 = np.logspace(-4, 1.5, 500)

grain_sizes = [0.2e-3, 1e-3, 5e-3, 20e-3]  # m
grain_labels = ['0.2 mm', '1 mm', '5 mm', '20 mm']
colors5 = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig, ax = plt.subplots(figsize=(7, 6))

for d_val, lab, col in zip(grain_sizes, grain_labels, colors5):
    e_total = edot_composite(tau5, T_fig5, d_val)
    ax.loglog(tau5, e_total, '-', color=col, lw=2, label=f'$d = $ {lab}')

# Dislocation creep (grain-size independent) for reference
ax.loglog(tau5, edot_disl(tau5, T_fig5), 'k--', lw=1, alpha=0.5,
          label='Dislocation creep only')

ax.set_xlabel('Effective deviatoric stress $\\tau_e$ (MPa)', fontsize=12)
ax.set_ylabel('Effective strain rate $\\dot{\\varepsilon}_e$ (s$^{-1}$)', fontsize=12)
ax.set_xlim(1e-4, 30)
ax.set_ylim(1e-18, 1e-2)
ax.legend(loc='upper left', fontsize=9, frameon=False)
ax.set_title(f'Effect of grain size ($T = {T_fig5:.0f}$ K)', fontsize=12)

fig.tight_layout()
fig.savefig('figures/fig05_grain_size_effect.pdf', bbox_inches='tight')
fig.savefig('figures/fig05_grain_size_effect.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure 5 saved.')